# Inspecting the learned BPE merges

CFPB-trained tokenizer at V = 8k (`bpe/artifacts/cfpb_8k`). Prints tokens only, never complaint text.

In [1]:
# setup: run from the repository root (python notebooks/build.py)
import sys; sys.path.insert(0, '.')

In [2]:
from bpe.tokenizer import Tokenizer
cfpb = Tokenizer.load('bpe/artifacts/cfpb_8k')
gen = Tokenizer.load('bpe/artifacts/general_8k')
first = len(cfpb.specials) + 256
print('first 30 merges (CFPB):   ', [cfpb.token_text(i) for i in range(first, first + 30)])
print('first 30 merges (WikiText):', [gen.token_text(i) for i in range(first, first + 30)])

first 30 merges (CFPB):    ['▁t', '▁a', '▁th', 're', 'in', '▁the', 'on', '▁c', 'ed', '▁w', '▁m', 'at', 'er', 'or', 'en', '▁s', 'it', '▁I', '▁to', '▁an', 'ou', 'is', '▁f', '▁b', '▁re', '▁p', '▁d', 'an', 'ing', '▁h']
first 30 merges (WikiText): ['▁t', 'he', '▁a', 'in', '▁the', 'er', 'on', '▁,', 're', '▁s', 'ed', '▁w', '▁o', 'nd', '▁.', 'at', 'or', 'en', '▁c', 'it', '▁f', 'is', 'es', 'ar', '▁b', '▁of', 'an', '▁in', 'al', 'ing']


## Longest tokens: what earned a whole vocabulary slot

In [3]:
toks = [cfpb.token_text(i) for i in range(first, len(cfpb))]
for t in sorted(toks, key=len, reverse=True)[:25]:
    print(repr(t))

'________________________________'
'▁misrepresentation'
'▁responsibilities'
'▁representatives'
'________________'
'▁reinvestigation'
'****************'
'▁acknowledgement'
'▁representative'
'▁correspondence'
'▁responsibility'
'▁communications'
'▁identification'
'▁unprofessional'
'▁electronically'
'▁Representative'
'▁representation'
'▁investigations'
'▁discrimination'
'▁Responsibility'
'▁aforementioned'
'▁misinformation'
'▁discriminatory'
'▁accountability'
'▁misrepresented'


## Tokens that are artifacts of how complaints are written

In [4]:
for t in ['▁1681', '▁USC', '▁pursuant', '▁hereby', '▁Navient', '▁Ocwen', '▁Comenity', '▁Synchrony']:
    print(f'{t:12s} CFPB vocab: {t in cfpb.vocab or t.replace("▁", "Ġ") in cfpb.vocab}')
print([t for t in toks if set(t) <= set('_*-=')][:5])

▁1681        CFPB vocab: True
▁USC         CFPB vocab: True
▁pursuant    CFPB vocab: True
▁hereby      CFPB vocab: True
▁Navient     CFPB vocab: True
▁Ocwen       CFPB vocab: True
▁Comenity    CFPB vocab: True
▁Synchrony   CFPB vocab: True
['__', '**', '____', '****', '________']


## Same sentence, two tokenizers (synthetic sentence)

In [5]:
s = 'I requested forbearance but the servicer reported me delinquent to Equifax under the FDCPA.'
print('CFPB    ', len(cfpb.encode(s)), cfpb.pieces(s))
print('WikiText', len(gen.encode(s)), gen.pieces(s))

CFPB     15 ['I', '▁requested', '▁forbearance', '▁but', '▁the', '▁servicer', '▁reported', '▁me', '▁delinquent', '▁to', '▁Equifax', '▁under', '▁the', '▁FDCPA', '.']
WikiText 28 ['I', '▁requested', '▁for', 'b', 'ear', 'ance', '▁but', '▁the', '▁serv', 'icer', '▁reported', '▁me', '▁del', 'in', 'qu', 'ent', '▁to', '▁E', 'qu', 'if', 'ax', '▁under', '▁the', '▁F', 'D', 'C', 'PA', '.']


## Redaction tokens stay atomic

In [6]:
s = 'On <DATE> they charged <AMT> to my account <REDACTED>.'
print(cfpb.pieces(s))

['On', '▁', '<DATE>', '▁they', '▁charged', '▁', '<AMT>', '▁to', '▁my', '▁account', '▁', '<REDACTED>', '.']
